След това бяха тествани подходи, базирани на автоенкодери[121], при които векторното представяне на слайдовете се извлича в латентното пространство, а сходството се изчислява чрез косинусова близост.

In [ ]:
# LOAD the data

In [4]:
import pandas as pd

# Specify the path to your CSV file
file_path = "../data/dataset_new/tainning_dataset.csv"

# Read the CSV file into a DataFrame
df = pd.read_csv(file_path)

df = df.sample(frac=1, random_state=42).reset_index(drop=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)


# Display the first few rows of the DataFrame
print(df.head())

# Assuming the DataFrame `df` has the same structure
image_pairs = list(zip(df['img1_path'], df['img2_path']))
style_labels = df['style_label'].tolist()
font_labels = df['font_label'].tolist()

# Now `image_pairs`, `style_labels`, and `font_labels` are reconstructed

df['original_index'] = df.index

                                           img1_path  \
0                      ../data/dataset/img1_3965.png   
1  ../data/dataset_new/data_complicated/img1_2574...   
2                      ../data/dataset/img1_4271.png   
3                       ../data/dataset/img1_336.png   
4                      ../data/dataset/img1_1981.png   

                                           img2_path  style_label  font_label  \
0                      ../data/dataset/img2_3965.png            0           0   
1  ../data/dataset_new/data_complicated/img2_2574...            0           0   
2                      ../data/dataset/img2_4271.png            1           1   
3                       ../data/dataset/img2_336.png            0           1   
4                      ../data/dataset/img2_1981.png            1           1   

   original_index  
0            3163  
1            1868  
2             102  
3             873  
4            2750  


In [5]:
import numpy as np
from PIL import Image # Import the PIL library

# Convert to numpy arrays
image_pairs = np.array(image_pairs)
style_labels = np.array(style_labels)
font_labels = np.array(font_labels)

# Function to load and preprocess an image
def load_and_preprocess_image(image_path):
    img = Image.open(image_path)
    img = img.resize((224, 224))  # Resize to match input shape
    img = np.array(img) / 255.0  # Normalize pixel values
    return img

# Load and preprocess the images
images_1 = np.array([load_and_preprocess_image(path) for path in image_pairs[:, 0]])
images_2 = np.array([load_and_preprocess_image(path) for path in image_pairs[:, 1]])


In [6]:
len(font_labels)

6000

In [ ]:
# MODEL training

In [7]:

from sklearn.model_selection import train_test_split

# Split the data into training (60%) and a temporary set (40%)
X_train_1, X_temp_1, y_style_train, y_style_temp = train_test_split(images_1, style_labels, test_size=0.4, random_state=42)
X_train_2, X_temp_2, y_font_train, y_font_temp = train_test_split(images_2, font_labels, test_size=0.4, random_state=42)

# Split the temporary set into validation (20%) and test sets (20%)
X_val_1, X_test_1, y_style_val, y_style_test = train_test_split(X_temp_1, y_style_temp, test_size=0.5, random_state=42)
X_val_2, X_test_2, y_font_val, y_font_test = train_test_split(X_temp_2, y_font_temp, test_size=0.5, random_state=42)



In [8]:
X_train_1.shape

(3600, 224, 224, 3)

In [9]:
def build_autoencoder(input_shape=(224, 224, 3), latent_dim=128):
    # Encoder
    input_img = Input(shape=input_shape)
    x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(input_img)
    x = layers.MaxPooling2D((2, 2), padding='same')(x)  # 112x112
    x = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(x)
    x = layers.MaxPooling2D((2, 2), padding='same')(x)  # 56x56
    x = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(x)
    x = layers.MaxPooling2D((2, 2), padding='same')(x)  # 28x28
    x = layers.Flatten()(x)
    encoded = layers.Dense(latent_dim, activation='relu')(x)

    # Decoder
    x = layers.Dense(28 * 28 * 128, activation='relu')(encoded)
    x = layers.Reshape((28, 28, 128))(x)
    x = layers.Conv2DTranspose(128, (3, 3), strides=2, padding='same', activation='relu')(x)  # 56x56
    x = layers.Conv2DTranspose(64, (3, 3), strides=2, padding='same', activation='relu')(x)   # 112x112
    x = layers.Conv2DTranspose(32, (3, 3), strides=2, padding='same', activation='relu')(x)   # 224x224
    decoded = layers.Conv2D(3, (3, 3), activation='sigmoid', padding='same')(x)  # 224x224x3

    autoencoder = models.Model(input_img, decoded)
    encoder = models.Model(input_img, encoded)
    
    return autoencoder, encoder


In [13]:
import numpy as np
import time
from sklearn.model_selection import KFold
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Input
from tensorflow.keras import layers

from tensorflow.keras import models


# Настройки
num_epochs = 100
batch_size = 16
learning_rate = 3e-4
num_folds = 5

# Обединяване на тренировъчните и валидационни данни
X_data = np.concatenate([X_train_1, X_train_2, X_val_1, X_val_2], axis=0)

# Стартиране на измерване на времето
start_time = time.time()

# Крос-валидация
kf = KFold(n_splits=num_folds, shuffle=True, random_state=42)

fold_no = 1
for train_index, val_index in kf.split(X_data):
    print(f"\n🔁 Fold {fold_no}/{num_folds}")

    X_train_fold, X_val_fold = X_data[train_index], X_data[val_index]

    # Създаване и компилиране на автоенкодера
    autoencoder, encoder = build_autoencoder()
    autoencoder.compile(optimizer='adam', loss='mse')

    # Early stopping
    early_stop = EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True
    )

    # Обучение
    history = autoencoder.fit(
        X_train_fold, X_train_fold,
        epochs=num_epochs,
        batch_size=batch_size,
        validation_data=(X_val_fold, X_val_fold),
        callbacks=[early_stop],
        verbose=1
    )

    fold_no += 1

# Измерване на продължителността
end_time = time.time()
duration = end_time - start_time
print(f"\n⏱️ Обучението с 5-кратна крос-валидация завърши за {duration:.2f} секунди ({duration/60:.2f} минути).")



🔁 Fold 1/5


InternalError: Failed copying input tensor from /job:localhost/replica:0/task:0/device:CPU:0 to /job:localhost/replica:0/task:0/device:GPU:0 in order to run _EagerConst: Dst tensor is not initialized.

In [ ]:
import time

# Build the autoencoder
autoencoder, encoder = build_autoencoder()

# Compile the model
autoencoder.compile(optimizer='adam', loss='mse')

# Combine training and validation data
X_train_combined = np.concatenate([X_train_1, X_train_2], axis=0)
X_val_combined = np.concatenate([X_val_1, X_val_2], axis=0)

# Track start time
start_time = time.time()

# Train the model
autoencoder.fit(
    X_train_combined, X_train_combined,
    epochs=20,
    batch_size=32,
    validation_data=(X_val_combined, X_val_combined)
)

# Track end time
end_time = time.time()

# Print training duration
duration = end_time - start_time
print(f"\n⏱️ Autoencoder training completed in {duration:.2f} seconds ({duration/60:.2f} minutes).")


In [ ]:
latent_vectors_1 = encoder.predict(X_test_1)
latent_vectors_2 = encoder.predict(X_test_2)


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

similarities = cosine_similarity(latent_vectors_1, latent_vectors_2)
pairwise_similarities = np.diag(similarities)  # similarity for corresponding test pairs


In [ ]:

# !pip install nbimporter --quiet  # only if not already installed

# import nbimporter
# from evaluation_module import evaluate_model_binary


# # Identify False Positives
# # false_positive_indices = [
# #     test_indices[i] for i, (true, pred) in enumerate(zip(y_test, style_pred_val_binary)) if true == 0 and pred == 1
# # ]

# results = evaluate_model_binary(
#     pred_val=pairwise_similarities,
#     y_test=y_style_test,
#     threshold=0.60
# )


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, jaccard_score,
    cohen_kappa_score, roc_auc_score, roc_curve, confusion_matrix
)
import numpy as np

def evaluate_model_binary(pred_val, y_test, test_indices=None, threshold=0.60, df=None):

    # Predict
    style_pred_val_binary = (pred_val > threshold).astype(int)

    # Identify False Positives
    false_positive_indices = []
    if test_indices is not None:
        false_positive_indices = [
            test_indices[i] for i, (true, pred) in enumerate(zip(y_test, style_pred_val_binary))
            if true == 0 and pred == 1
        ]

    if df is not None and false_positive_indices:
        for idx in false_positive_indices:
            attributes_img1 = json.loads(df.loc[idx, 'attributes_img1'])
            attributes_img2 = json.loads(df.loc[idx, 'attributes_img2'])
            print(f"False Positive Pair {idx}:")
            print(f"Attributes Image 1: {attributes_img1}")
            print(f"Attributes Image 2: {attributes_img2}")

    # Metrics
    results = {
        'accuracy': accuracy_score(y_test, style_pred_val_binary),
        'precision': precision_score(y_test, style_pred_val_binary),
        'recall': recall_score(y_test, style_pred_val_binary),
        'f1_score': f1_score(y_test, style_pred_val_binary),
        'jaccard': jaccard_score(y_test, style_pred_val_binary),
        'kappa': cohen_kappa_score(y_test, style_pred_val_binary),
        'roc_auc': roc_auc_score(y_test, pred_val),
    }

    # Print metrics
    print(f"Accuracy: {results['accuracy']:.2f}")
    print(f"Precision: {results['precision']:.2f}")
    print(f"Recall: {results['recall']:.2f}")
    print(f"F1 Score: {results['f1_score']:.2f}")
    print(f"Jaccard Similarity: {results['jaccard']:.2f}")
    print(f"Cohen's Kappa: {results['kappa']:.2f}")
    print(f"ROC AUC: {results['roc_auc']:.2f}")

    # ROC curve
    fpr, tpr, thresholds = roc_curve(y_test, pred_val)
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, label=f'ROC Curve (AUC = {results["roc_auc"]:.2f})')
    plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
    plt.title("ROC Curve")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.legend(loc="lower right")
    plt.grid(True)
    plt.show()

    # Confusion matrix
    cm = confusion_matrix(y_test, style_pred_val_binary)
    plt.figure(figsize=(6, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=["Predicted: 0", "Predicted: 1"],
                yticklabels=["Actual: 0", "Actual: 1"])
    plt.title("Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()

    return results

########EXAMPLE USAGE
results = evaluate_model_binary(
     pred_val=pairwise_similarities,
      y_test=y_style_test,
     threshold=0.60
     )

